In questa sezione si introduce il **Probabilistic Information Retrieval**: i documenti non vengono ordinati per somiglianza alla query, ma **per la probabilità che siano rilevanti**.

Parlando del Vector Space Model finora abbiamo visto il tutto in modo piuttosto empirico: i documenti e le query si rappresentano come vettori di termini, dove ognuno è pesato tf-idf, e si ordinano per cosine similarity. Abbiamo detto che il modello funziona bene, ma non abbiamo mai veramente spiegato perché sia una buona idea rappresentare i documenti in questo modo e **dare un peso maggiore a termini rari**. Il Probabilistic Information Retrieval ci dà una spiegazione più formale di questo modello, e ci permette di capire meglio perché funziona.

## Perché Probabilistic IR?
Il motivo del Probabilistic IR è che in IR la rilevanza di un documento è una variabile aleatoria: quando un utente fa una query, il sistema **non capisce veramente il bisogno informativo dell'utente**, ma ha solo una rappresentazione approssimativa di esso (i termini nella query). Allo stesso modo i documenti non sono compresi nella loro interezza, ma sono trasformati in una rappresentazione (es. vettori di termini, con pesi tf-idf etc..). Quindi in realtà il sistema quando risponde a una query sta rispondendo a una domanda incerta: **qual è la probabilità che un documento sia rilevante per una query?** $$Pr(d is relevant | q)$$

Tra le fonti di aleatorietà ci sono Utenti, Documenti e Sistema di IR:
- diversi utenti possono avere bisogni informativi diversi, anche se fanno la stessa query
- diversi utenti possono pensare che un documento sia rilevante o meno, anche a parità di bisogno informativo
- i documenti sono rappresentati in modo approssimativo -> perdita di informazione
- il sistema IR stesso può introdurre approssimazioni/errori (es. errori di tokenizzazione, stemming, etc..) nello stimare la rilevanza di un documento

Il modello probabilistico vuole rispondere **a qualcosa di molto più grande rispetto al Vector Space Model**. Mentre nel Vector Space Model i documenti sono ordinati in base alla loro somiglianza alla query, in realtà simile non significa necessariamente rilevante (un documento può contenere molte parole della query, ma non rispondere al bisogno informativo dell'utente). 

Il modello probabilistico invece **prova a formalizzare direttamente quello che vogliamo davvero: dare all'utente documenti rilevanti**.

Vedremo come anticipato come, partendo da questa idea molto pretenziosa, si arrivi con le dovute semplificazioni proprio al Vector Space Model, dimostrando come quest'ultimo sia un buon modello per stimare la probabilità di rilevanza.

Vedremo due principali modelli probabilistici:
- **Classical Probabilistic Retrieval Model**: l'idea è, dato un documento e una query, stimare la **probabilità che il documento sia rilevante per la query**. Questo filone si basa sul **Probability Ranking Principle** (principio secondo cui i documenti vanno ordinati in base alla loro probabilità di rilevanza). I modelli concreti che provano a calcolare questa probabilità sono:
  1. **Binary Independence Model (BIM)**: in questo modello si assume che i documenti siano rappresentati come vettori binari (0/1), dove 1 indica la presenza di un termine e 0 la sua assenza. 
  2. **BestMatch25 (BM25 o Okapi)**: è un modello più sofisticato che tiene conto non solo della presenza/assenza dei termini, ma anche della loro frequenza e della lunghezza del documento. BM25 è uno dei modelli più usati in IR, ed è alla base di molti motori di ricerca moderni.
- **Language Models for IR**: qui invece si ragiona al contrario, volendo stimare **qual è la probabilità che un documento "abbia generato" la query**.

## Alcune nozioni base
**Chain Rule**: $$Pr(A, B) = Pr(A|B) \cdot Pr(B) = Pr(B|A) \cdot Pr(A)$$
**Complemento**: $$Pr(\bar{A}, B) = Pr(B|\bar{A}) \cdot Pr(\bar{A})$$
**Probabilità totale**: $$Pr(B) = Pr(A, B) + Pr(\bar{A}, B)$$
**Bayes' Theorem**: $$Pr(A|B) = \frac{Pr(B|A) \cdot Pr(A)}{Pr(B)}$$
**Odds**: $$O(A) = \frac{Pr(A)}{Pr(\bar{A})} = \frac{Pr(A)}{1 - Pr(A)}$$

Odds è particolarmente utile in quanto mostra diverse casistiche:
- Se $Pr(A) = 0.5$, allora $O(A) = 1$ (probabilità uguale a quella del complemento, non possiamo dire nulla)
- Se $Pr(A) > 0.5$, allora $O(A) > 1$ (il documento è più probabile che sia rilevante che non rilevante)
- Se $Pr(A) < 0.5$, allora $O(A) < 1$ (il documento è più probabile che non sia rilevante)

## Introduction
Per implementare un sistem di IR basato su questo modello dobbiamo anzitutto **assumere che la rilevanza sia una variabile aleatoria binaria**: un documento è o rilevante o non rilevante per una query:
$$R_{d,q} = \begin{cases}
1 & \text{se il documento } d \text{ è rilevante per la query } q \\
0 & \text{altrimenti}
\end{cases}$$
Nel caso del classical probabilistic retrieval model, vogliamo stimare la probabilità che un documento sia rilevante per una query, ovvero:
$$Pr(R_{d,q} = 1 | d, q)$$
Ma questa probabilità deve dipendere anche dagli utenti, dal momento che come anticipato la rilevanza è soggettiva, quindi assumendo che tutti gli utenti siano indipendenti, usiamo la probabilità totale:
$$Pr(R | d, q) = \sum_{u \in U} Pr(R | d, q, u) \cdot Pr(u)$$
dove $U$ è l'insieme di tutti gli utenti.

Si **Assumerà in generale che tutti gli utenti siano ugualmente probabili**, quindi alla fine si considera in realtà solo $Pr(R | d, q)$.

Un documento sarà restituito solo se è più probabile che sia rilevante che non rilevante, ovvero:
$$O(R | d, q) = \frac{Pr(R | d, q)}{Pr(\bar{R} | d, q)} > 1$$

Tuttavia **non abbiamo informazioni sui giudizi di rilevanza da parte degli utenti per ogni documento e query (sarebbe impossibile, miliardi di combinazioni) -> sfruttiamo Bayes, due metodi**: 
- **Metodo 1 (Classical Probabilistic Retrieval Model)**:  
  $$Pr(R \mid d,q)=\frac{Pr(d \mid R,q)\cdot Pr(R \mid q)}{Pr(d \mid q)}$$

  cioè stimiamo quanto è probabile che il documento sia rilevante guardando quanto è probabile osservare quel documento tra i documenti rilevanti per la query fissata \(q\).

  Analogamente:

  $$Pr(\bar{R} \mid d,q)=\frac{Pr(d \mid \bar{R},q)\cdot Pr(\bar{R} \mid q)}{Pr(d \mid q)}$$

  cioè stimiamo quanto è probabile che il documento non sia rilevante guardando quanto è probabile osservare quel documento tra i documenti non rilevanti per la stessa query \(q\).
- **Metodo 2 (language models for IR)**:  
  $$Pr(R \mid d,q)=\frac{Pr(q \mid R,d)\cdot Pr(R \mid d)}{Pr(q \mid d)}$$

  cioè: stimiamo la probabilità che il documento sia rilevante guardando quanto è probabile osservare la query a partire dal documento (assumendo che sia rilevante, R).

  Analogamente:

  $$Pr(\bar{R} \mid d,q)=\frac{Pr(q \mid \bar{R},d)\cdot Pr(\bar{R} \mid d)}{Pr(q \mid d)}$$

  cioè: stimiamo la probabilità che il documento non sia rilevante guardando quanto è probabile osservare la query a partire da un documento non rilevante. 



### Classical Probabilistic Retrieval Model
Ci focalizziamo quindi sul Metodo 1.
$$Pr(R \mid d,q)=\frac{Pr(d \mid R,q)\cdot Pr(R \mid q)}{Pr(d \mid q)}$$

Sappiamo che in generale per un sistema IR, data una collezione di documenti e una query, si vuole restituire una **lista ordinata di documenti**. Vedremo con il **Probability Ranking Principle** che questa lista ordinata deve essere ordinata in base alla probabilità di rilevanza, ovvero $Pr(R \mid d,q)$.

Vediamo ora di semplificare questa formula, facendo alcune assunzioni.

Anzitutto si fa una **Assunzione di Indipendenza: la rilevanza di un documenti è indipendente dagli altri documenti**. In questo modo possiamo stimare separatamente $Pr(R \mid d_1, q)$, $Pr(R \mid d_2, q)$, etc.. senza dover considerare combinazioni del tipo $Pr(R_{d_1, d_2} | q)$ (se ho già messo un documento nella lista, non cambia la probabilità che un altro documento sia rilevante o meno).

Dopodiché si **Assume che documento e query siano indipendenti** -> $Pr(d \mid q) = Pr(d)$. In questo modo la formula diventa:
$$Pr(R \mid d,q)=\frac{Pr(d \mid R,q)\cdot Pr(R \mid q)}{Pr(d)}$$
dove:
- $Pr(d \mid R,q)$: è la probabilità di osservare il documento \(d\) tra i documenti rilevanti per la query \(q\).
- $Pr(R \mid q)$: ci dice quanto è probabile che un documento sia rilevante per la query \(q\) (indipendentemente dal documento specifico).
- $Pr(d)$: è la probabilità di osservare il documento \(d\) in generale, indipendentemente dalla query e dalla rilevanza.

Si fa lo stesso ovviamente per $Pr(\bar{R} \mid d,q)$.

Si fanno poi le seguenti **ulteriori assunzioni**:
- **Probabilità uniforme dei documenti**: $Pr(d) = Pr(d')$ (tutti i documenti sono ugualmente probabili) -> non serve nel ranking, è una costante uguale per tutti i documenti
- $Pr(R \mid q)$ **è costante** (data una query, la probabilità che un documento sia rilevante è la stessa per tutti i documenti) -> non serve nel ranking, è una costante uguale per tutti i documenti

**Ai fini del ranking posso perciò ignorare i termini costanti rispetto ai documenti; quindi la formula si riduce a una funzione proporzionale a** $Pr(d \mid R,q)$. Più avanti, confrontando rilevanti e non rilevanti, si userà il rapporto tra $Pr(d \mid R,q)$ e $Pr(d \mid \bar R,q)$ per determinare la posizione nel ranking di un documento (Odds).

Inoltre si assume rilevanza binaria, per cui $Pr(\bar R \mid d,q)+Pr(R \mid d,q)=1$. 

#### Probability Ranking Principle PRP
Ma perché si vuole ordinare i documenti in base a $Pr(R \mid d, q)$ ? Perché il PRP dice che **se si ordinano i documenti per probabilità di rilevanza, si ottiene il miglior risultato possibile**. Quindi questo è il miglior ranking possibile.

Cerchiamo di capire intuitivamente il perché, formalizzando cosa significa "miglior ranking". Definiamo due possibili errori: $C(d, q)$ il **costo di non restituire un documento che era rilevante** (falso negativo) e $C'(d, q)$ il **costo di restituire un documento che non era rilevante** (falso positivo). 

Allora se il sistema restituisce un insieme di documenti $D(q)$, il **rischio/costo atteso** è: $$R(D(q)) = \sum_{d \in D(q)} Pr(\bar{R} | d, q) \cdot C'(d, q) + \sum_{d \notin D(q)} Pr(R | d, q) \cdot C(d, q)$$
ossia per i documenti restituiti rischio di sbagliare se non sono rilevanti mentre per quelli che non sono restituiti rischio di sbagliare se sono rilevanti.

Assumiamo ora di trovarci nel caso più semplice, in cui $C(d, q) = C'(d, q) = 1$ (costo unitario per entrambi gli errori). Allora il rischio atteso diventa: $$R(D(q)) = \sum_{d \in D(q)} (1 - Pr(R | d, q)) + \sum_{d \notin D(q)} Pr(R | d, q)$$
ossia se restituisco un documento pago l'errore se non è rilevante altrimenti se non lo restituisco pago l'errore se è rilevante.

Sta proprio in questa formula finale la dimostrazione del PRP: **se devo restituire i migliori k documenti, allora il rischio atteso è minimo proprio quanto scelgo i k documenti con la più alta probabilità di rilevanza**.

Quindi con funzione di costo 0/1, il PRP è ottimale perché minimizza la perdita di attesa. **Tuttavia bisogna fare attenzione: questa ottimalità è vera solo dal momento che le probabilità di rilevanza sono stimate correttamente -> il problema diventa stimare correttamente $Pr(R | d, q)$**. Se le stime sono errate, il PRP non è più ottimale.


#### Binary Independence Model (BIM)
Come calcolare quindi $Pr(R \mid d, q)$? Servono dei modelli per stimarle, quello più semplice è il Binary Independence Model (BIM). 

In BIM si fanno **due assunzioni importanti**:
1. **La rilevanza di un documento è indipendente dagli altri documenti** 
2. **La rilevanza è booleana: o un documento è rilevante o non lo è** (non ci sono gradi di rilevanza, è una variabile binaria)

Per prima cosa dobbiamo dare una rappresentazione matematica a un documento d. Un documento sarà **un vettore di feature**: $d = (f_1, f_2, ..., f_m)$ dove ogni $f_i$ indica la presenza di un termine (1 se il termine è presente, 0 altrimenti). 

Allora $Pr(R \mid d, q) = Pr(R \mid f_1, f_2, ..., f_m, q)$, e per quanto detto in precedenza considerando le semplificazioni per il ranking: $$Pr(R \mid f_1, f_2, ..., f_m, q) \propto Pr(f_1, f_2, ..., f_m \mid R,q)$$ Ricordiamo che poi ai fini del ranking questa probabilità sarà confrontata con il caso dei documenti non rilevanti, cioè con $Pr(f_1, f_2, ..., f_m \mid \bar{R},q)$.

Quindi in pratica adesso stiamo cercando di capire quanto le feature (presenza/assenza dei termini) siano indicative della rilevanza del documento. Tuttavia farlo direttamente è molto difficile: fissato un documento conosco le sue m feature, ma stimare la probabilità di osservare proprio quelle m feature tra i documenti rilevanti per la query è molto complesso.

**Si fa quindi l'assunzione fondamentale Naive Bayes: si assume che le feature (assenza/presenza di un termine) siano indipendenti tra loro**. Si osservi che questa è un'assunzione forte ma non necessariamente vera (es. presenza di python e presenza di code sicuramente sono correlate), però semplifica molto l'analisi. In questo modo si ottiene: $$Pr(f_1, f_2, ..., f_m \mid R,q) = \prod_{i=1}^m Pr(f_i \mid R,q)$$

Con BIM si torna alla **rappresentazione vettoriale booleana dei documenti**: ogni documento è un vettore binario nello spazio ${0,1}^{m}$, dove m è il numero di termini distinti nella collezione. $f_i$ è 1 sse il termine $t_i$ è presente nel documento, 0 altrimenti. Vale lo stesso per la query. Pertanto non stiamo più stimando $Pr(R \mid d, q)$, ma $Pr(R \mid v_d, v_q)$, ossia la probabilità di rilevanza date le rappresentazioni vettoriali binarie di documento e query.

Tornando di nuovo al ranking, si ottiene quindi che vogliamo calcolare le seguenti probabilità (con =rank si intende proporzionale):

<img src="img/diocane.png" alt="formula" width="400"/>

Dove $Pr(v_d \mid R, v_q)$ e $Pr(v_d \mid \bar{R}, v_q)$ sono le probabilità che se un documento rilevante/non rilevante è restituito per la query -> quel documento ha rappresentazione vettoriale $v_d$. Per stimare ste probabilità si usano statistiche legate alla collezione.

##### Derivazione della formula di ranking
Deriviamo concretamente sta formula di ranking, partendo da $Pr(R \mid v_d, v_q)$. Come anticipato per fare ranking usiamo gli odds, quindi vogliamo calcolare: $$O(R \mid v_d, v_q) = \frac{Pr(R \mid v_d, v_q)}{Pr(\bar{R} \mid v_d, v_q)}$$
Applicando Bayes e facendo le assunzioni viste per il ranking si ottiene in definitiva che $$O(R \mid v_d, v_q) \propto \frac{Pr(v_d \mid R, v_q)}{Pr(v_d \mid \bar{R}, v_q)}$$
Si applica poi Naive bayes:
$$O(R \mid v_d, v_q) \propto \prod_{i=1}^m \frac{Pr(x_i \mid R, v_q)}{Pr(x_i \mid \bar{R}, v_q)}$$
dove $x_i$ è la i-esima feature (presenza/assenza del termine $t_i$ nel documento $d$).

A questo punto separiamo i termini presenti da quelli assenti nel documento:
$$O(R \mid v_d, v_q) = \prod_{i:x_i=1} \frac{Pr(x_i=1 \mid R, v_q)}{Pr(x_i=1 \mid \bar{R}, v_q)} \prod_{i:x_i=0} \frac{Pr(x_i=0 \mid R, v_q)}{Pr(x_i=0 \mid \bar{R}, v_q)}$$
Assumiamo ora ragionevolmente che **per quanto riguarda i termini che non fanno parte della query, questi non aiutano a distinguere se il documento è rilevante o meno -> $Pr(x_i=1 \mid R, v_q) = Pr(x_i=1\mid \bar{R}, v_q)$** e lo stesso per $x_i=0$. Quindi il rapporto diventa 1 per i termini non presenti nella query, suppondendo y_i indichi il termine della query si ottiene:
$$O(R \mid v_d, v_q) = \prod_{x_i=y_i=1} \frac{Pr(x_i=1 \mid R, v_q)}{Pr(x_i=1 \mid \bar{R}, v_q)} \prod_{x_i=0, y_i=1} \frac{Pr(x_i=0 \mid R, v_q)}{Pr(x_i=0 \mid \bar{R}, v_q)}$$
Per pulire la notazione, definiamo come $p_t = Pr(x_t = 1 \mid R, v_q)$ la probabilità che un termine x_t sia presente in un documento rilevante per la query e $u_t = Pr(x_t = 1 \mid \bar{R}, v_q)$ la probabilità che un termine x_t sia presente in un documento non rilevante per la query. Di seguito la tabella di contingenza che mostra i relativi significati:

<img src="img/contingenza.png" alt="tabella di contingenza" width="500"/>

Abbiamo quindi $$O(R \mid v_d, v_q) = \prod_{x_i=y_i=1} \frac{p_i}{u_i} \prod_{x_i=0, y_i=1} \frac{1-p_i}{1-u_i}$$
Abbiamo due produttorie, uno dipendende da $x_i=1$ e l'altro da $x_i=0$, vorremmo trovarci un unico prodotto sui termini presenti sia nella query che nel documento. Quindi per ogni termine $x_i=y_i=1$ moltiplico per $1 = \frac{1-p_i}{1-u_i} \cdot \frac{1-u_i}{1-p_i}$, mettendo la prima frazione nel secondo prodotto e la seconda frazione nel primo prodotto. 

**Il secondo prodotto in $O(R \mid v_d, v_q)$ rappresentava tutti i termini presenti nella query ma non nel documento**, stiamo aggiungendo un valore $\frac{1-p_i}{1-u_i}$ che invece conta **per i termini presenti sia nella query che nel documento -> si ottiene qualcosa che che contiene tutti i termini presenti nella query indipendentemente dal documento** $\prod_{y_i=1} \frac{1-p_i}{1-u_i}$. In definitiva si ottiene $$O(R \mid v_d, v_q) = \prod_{x_i=y_i=1} \frac{p_i (1-u_i)}{u_i (1-p_i)} \prod_{y_i=1} \frac{1-p_i}{1-u_i}$$

Ma ora il secondo prodotto dipende solo dalla query e non dal documento -> è una costante nel ranking, quindi si può ignorare. Quindi si arriva a $$O(R \mid v_d, v_q) \propto \prod_{x_i=y_i=1} \frac{p_i (1-u_i)}{u_i (1-p_i)}$$

Poiché è più facile lavorare con le sommatorie piuttosto che con le produttorie, si prende il logaritmo di $O(R \mid v_d, v_q)$, ottenendo il **Retrieval Status Value RSV del documento**: $$RSV_d = \sum_{x_i=y_i=1} \log \frac{p_i (1-u_i)}{u_i (1-p_i)}$$

In definitiva,**RSV_d è lo score finale nel BIM usato per ordinare i documenti**: più è alto, più è probabile che il documento sia rilevante per la query.

Sia $c_i = \log \frac{p_i (1-u_i)}{u_i (1-p_i)} = \log \frac{p_i}{1-p_i} - \log \frac{u_i}{1-u_i}$, allora $RSV_d = \sum_{x_i=y_i=1} c_i$. Si osserva esplicitamente come $c_i$ confronti due cose:
- $\frac{p_i}{1-p_i}$: sono gli odds che un termine i compaia in un documento rilevante per la query
- $\frac{u_i}{1-u_i}$: sono gli odds che un termine i compaia in un documento non rilevante per la query

Quindi $c_i$ misura proprio quanto un termine è più caratteristico nei documenti rilevanti rispetto a quelli non rilevanti:
- se $c_i > 0$ vuol dire che il termine i compare più facilmente nei documenti rilevanti -> è un termine utile e aumenta lo score
- se $c_i = 0$ vuol dire che il termine i compare con la stessa probabilità nei documenti rilevanti e non rilevanti -> non aiuta a distinguere
- se $c_i < 0$ vuol dire che il termine i compare più facilmente nei documenti non rilevanti -> è un termine "dannoso" per la rilevanza

**Ci accorgiamo quindi di quanto il Vector Space Model sia simile al BIM**: in entrambi i modelli i documenti sono rappresentati come vettori di termini, e inoltre si prendono in considerazione solo i termini presenti sia nella query che nel documento per il calcolo dello score. **La differenza sta in come vengono calcolati i pesi del termine**: mentre in VSM il peso è empirico, tipo tf-idf (prodotto scalare tra componenti in comune dei due vettori query e documento): $$score(d,q) = \sum_{t_i \in d \cap q} w_{i,d} \cdot w_{i,q}$$ 
Mentre in BIM il peso deriva come visto da una giustificazione probabilistica
$$RSV_d = \sum_{t_i \in d \cap q} c_i$$
In particolare è possibile usare per entrambi modelli l'inverted index, proprio perché **in entrambi i casi per calcolare lo score di un documento rispetto a una query, è necessario considerare solo i documenti che contengono almeno un termine della query** (quindi mi basta prendere in **OR** tutti i documenti che contengono almeno un termine della query, e poi calcolare lo score su quelli per poi restituirli ordinati).

##### Stima di $p_i$ e $u_i$
Abbiamo trovato quindi che con BIM per ordinare i documenti rispetto a una query è necessario conoscere $c_i = \log \frac{p_i (1-u_i)}{u_i (1-p_i)}$ per ogni termine i. Ma $p_i = Pr(x_i = 1 \mid R, v_q)$ e $u_i = Pr(x_i = 1 \mid \bar{R}, v_q)$ sono probabilità che non possiamo conoscere a priori, quindi dobbiamo stimarle.

Per stimare queste probabilità ci sono due casi da prendere in considerazione:
- **Caso 1: abbiamo giudizi di rilevanza**: gli utenti ci hanno etichettato i documenti come rilevanti o non rilevanti per la query oppure si hanno degli pseudo-feedback (il sistema usa modelli semplici tipo tf-idf per stimare la rilevanza come baseline, effettua una query e assume che i primi k documenti restituiti siano rilevanti)
- **Caso 2: non abbiamo nulla a cui far riferimento**: in tal caso dobbiamo fare ulteriori assunzioni (e vedremo come ciò ci porterà a idf)

###### Caso 1: abbiamo giudizi di rilevanza
Definiamo:
- $N$: numero totale di documenti nella collezione
- $R$: numero di documenti rilevanti per la query
- $df_i$: numero di documenti che contengono il termine i
- $r_i$: numero di documenti rilevanti che contengono il termine i

Si definisce la seguente tabella di contingenza per il numero di documenti (sopra documenti rilevanti/non rilevanti, a sx se il termine è presente o meno):

<img src="img/contingenza2.png" alt="tabella di contingenza 2" width="500"/>

dove ad esempio il numero totale di documenti non rilevanti che non contengono il termine i (basso a dx) è dato da $(N - df_i) - (R - r_i)$ (tutti i documenti meno quelli che contengono il termine i meno quelli rilevanti che non contengono il termine i).

A questo punto possiamo stimare le probabilità $p_i$ e $u_i$: 
$$p_i = Pr(x_i = 1 \mid R, v_q) \approx \frac{r_i}{R}$$
(tra i rilevanti, quanti contengono il termine r_i)
$$u_i = Pr(x_i = 1 \mid \bar{R}, v_q) \approx \frac{df_i - r_i}{N - R}$$
(tra i non rilevanti, quanti contengono il termine i)

Sostituendo il tutto nel peso si ottiene:

<img src="img/diocane2.png" alt="formula" width="300"/>

Problema: se $r_i = 0$ o $df_i - r_i = 0$ allora $c_i$ diventa infinito. Per evitare questo problema si usa la tecnica di **Smoothing**: si aggiunge una costante $\alpha$ (tipo 0.5) a tutti i contatori, si ottiene:

<img src="img/diocane3.png" alt="formula" width="300"/>

Esercizio: (da fare)

<img src="img/esercizio.png" alt="esercizio" width="500"/>

###### Caso 2: non abbiamo na mazza
Non possiamo quindi stimare $p_i$ e $u_i$ con i giudizi di rilevanza, ma possiamo fare delle assunzioni per stimarli comunque.

In questo caso **l'assunzione chiave è che i documenti rilevanti sono pochissimi rispetto all'intera collezione**, quindi $\bar{R} \approx N$. Secondo questo ragionamento $$u_i = Pr(x_i = 1 \mid \bar{R}, v_q) \approx \frac{df_i}{N}$$ ossia la probabilità che un termine i compaia in un documento non rilevante è approssimativamente la probabilità che compaia in un documento qualsiasi, dato che i documenti rilevanti sono pochi.

Ci manca ora $p_i = Pr(x_i = 1 \mid R, v_q)$. Poiché della probabilità che un termine compaia in un documento rilevante non sappiamo assolutamente nulla -> **facciamo l'assunzione brutale che $p_i = 0.5$ per ogni termine i** (non abbiamo motivo di credere che un termine sia più probabile di un altro nei documenti rilevanti, assumo probabilità neutra).

Quindi ricordando che $c_i = \log \frac{p_i (1-p_i)}{u_i (u_i)}$, il primo termine diventa $\log 1 = 0$. Sostituendo quindi solo il secondo, si ottiene: 
$$c_i = \log \frac{1 - \frac{df_i}{N}}{\frac{df_i}{N}} = \log \frac{N - df_i}{df_i}$$
Infine, **assumento che il numero di documenti contenenti il termine i sia molto più piccolo del numero totale di documenti** ($df_i << N$), si ottiene: $$c_i \approx \log \frac{N}{df_i}$$ ossia c_i è proprio l'idf del termine i!

Quindi con queste assunzioni BRUTALI l'RSV finale diventa $$RSV_d = \sum_{t_i \in d \cap q} \log \frac{N}{df_i}$$ cioè somma degli idf dei termini della query presenti nel documento.

(idf -> termine raro implica df piccolo e quindi peso idf alto -> termine significativo per spingere il documento in alto nel ranking)

Ancora più di prima quindi si osserva in questo caso **quanto BIM sia simile a VSM**: il peso stavolta è soltanto l'idf mentre in VSM è tf-idf. 

Si ricorda che in VSM lo score usato è la cosine similarity tra i due vettori, mentre in BIM non si usa la cosine similarity ma per calcolare lo score di un documento si usa la sommatora dei c_i, dove c_i è peso probabilistico. Operativamente è però simile a VSM perché anche qui si sommano i contributi dei temini in comune tra documento e query.

**La vera differenza è che VSM è basato su similarità geometrica tra vettori, mentre BIM è basato su una giustificazione probabilistica. Operativamente però entrambi usano vettori di termini e inverted index.**

**Limiti di BIM**: 
- **Rilevanza binaria**: BIM tiene conto solo se $x_i = 1$ (il termine i compare) o $x_i = 0$ (il termine i non compare), ma **non tiene conto di quanto un termine compare in un documento (tf)**.
- **BIM non tiene conto della lunghezza del documento**: un documento più lungo avrà più termini e quindi più probabilità di contenere i termini della query, ma BIM non tiene conto di questo aspetto.

Per superare questi problemi si passa quindi a **BM25**.